In [ ]:
from gettext import install
import pip

%pip install requests beautifulsoup4 pandas openpyxl lxml selenium


In [ ]:

# TennisAbstract ATP Elo

from bs4 import BeautifulSoup
import lxml
import requests
import pandas as pd
import time

# Scrape ATP Elo ratings - first 100 players
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}

url2 = 'https://tennisabstract.com/reports/atp_elo_ratings.html'
response = requests.get(url2, headers=headers)
response.raise_for_status()

soup = BeautifulSoup(response.text, 'lxml')


# Find table with id "reportable"
table = soup.find('table', {'id': 'reportable'})

if table:
    # Extract column headers from thead or first tr with th elements
    header_row = table.find('thead')
    if header_row:
        headers_list = [th.get_text(strip=True) for th in header_row.find_all('th')]
    else:
        # Fallback: try to find headers in the first row
        first_row = table.find('tr')
        if first_row and first_row.find_all('th'):
            headers_list = [th.get_text(strip=True) for th in first_row.find_all('th')]
        else:
            headers_list = None  # Will be handled later
    
    # Extract data rows (skip header row, get first 100 data rows)
    rows = []
    for tr in table.find_all('tr')[1:101]:
        cols = tr.find_all('td')
        if cols:  # Only process rows with td elements
            row_data = [col.get_text(strip=True) for col in cols]
            rows.append(row_data)
    
    # Create DataFrame
    if rows:
        # Generate default column names if headers weren't found
        if headers_list is None:
            headers_list = [f"Col_{i}" for i in range(len(rows[0]))]
        
        df_atp = pd.DataFrame(rows, columns=headers_list)
        
        # Optional: Display information (uncomment if needed)
        # print(df_atp)
        # print(f"\nColumn titles: {list(df_atp.columns)}")
        # print(f"\nTotal players scraped: {len(df_atp)}")
    else:
        print("No data rows found in table")
        df_atp = pd.DataFrame()  # Empty DataFrame
else:
    print("Table with id 'reportable' not found")
    df_atp = pd.DataFrame()  # Empty DataFrame

# Check for empty columns
#print("Empty or unnamed columns:")
#for i, col in enumerate(df_atp.columns):
#    if col == '' or col.strip() == '':
#        print(f"  Column {i}: Empty header")
#        print(f"    Sample values: {df_atp.iloc[:3, i].tolist()}")

# Keep only columns with data (remove empty columns)
df_atp_clean = df_atp.drop(df_atp.columns[[4, 11, 14]], axis=1)

# Remove spaces from column headers
df_atp_clean.columns = df_atp_clean.columns.str.replace('\xa0', '')

# Remove spaces and non-breaking spaces from Player column
df_atp_clean['Player'] = [name.replace(' ', '').replace('\xa0', '') for name in df_atp_clean['Player']]
print(df_atp_clean)

# Export to CSV
#df_atp_clean.to_csv('df_atp_elo.csv', index=False)
